In [1]:
from pathlib import Path

import polars as pl
from lynceus_core.features import MOLECULAR_FEATURES
from surrogate_model.common.model import ModelType
from surrogate_model.common.threshold import get_validation_threshold
from surrogate_model.common.training import train_and_evaluate

In [2]:
data_dir = Path("data")
rounds = [f"round_{i}" for i in range(1, 5)]

train_features = [str(data_dir / r / "features.parquet") for r in rounds]
train_labels = [str(data_dir / r / "docking.results.parquet") for r in rounds]

val_features = str(data_dir / "validation" / "features.parquet")
val_labels = str(data_dir / "validation" / "docking.results.parquet")

In [3]:
features = list(MOLECULAR_FEATURES)
threshold = get_validation_threshold(val_labels)

In [4]:
comparison = []
for model_type in (ModelType.LIGHTGBM, ModelType.MLP):
    _, res = train_and_evaluate(
        train_features_paths=train_features,
        train_labels_paths=train_labels,
        validation_features_path=val_features,
        validation_labels_path=val_labels,
        features=features,
        threshold=threshold,
        random_seed=1000,
        model_type=model_type,
    )
    comparison.append(
        pl.DataFrame(res["metrics"]).with_columns(
            pl.lit(str(model_type)).alias("model")
        )
    )

pl.concat(comparison)

{"label_bins": "array([62647, 14969])", "event": "dataset", "timestamp": "2026-11-26T21:41:37.505929Z", "level": "info"}
[LightGBM] [Info] Number of positive: 11956, number of negative: 50136
[LightGBM] [Info] Total Bins 211885
[LightGBM] [Info] Number of data points in the train set: 62092, number of used features: 4568
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.192553 -> initscore=-1.433506
[LightGBM] [Info] Start training from score -1.433506
{"model_type": "lightgbm", "samples": 62092, "duration_seconds": 26.027, "event": "Trained model", "timestamp": "2026-11-26T21:42:03.793100Z", "level": "info"}
{"dataset_name": "Initial Split", "metrics": "shape: (4, 7)\n\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u25

Top %,BEDROC (80.5),BEDROC (160.9),EF,Recall,Num active caught,Num active,model
f64,f64,f64,f64,f64,i64,i64,str
1.0,0.415402,0.377786,30.345399,0.303253,289,953,"""lightgbm"""
5.0,0.415402,0.377786,13.053927,0.652676,622,953,"""lightgbm"""
10.0,0.415402,0.377786,8.226912,0.822665,784,953,"""lightgbm"""
25.0,0.415402,0.377786,3.840625,0.960126,915,953,"""lightgbm"""
1.0,0.385706,0.345357,26.565349,0.265477,253,953,"""mlp"""
5.0,0.385706,0.345357,12.948992,0.647429,617,953,"""mlp"""
10.0,0.385706,0.345357,8.426289,0.842602,803,953,"""mlp"""
25.0,0.385706,0.345357,3.874204,0.96852,923,953,"""mlp"""


In [5]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from surrogate_model.common.data import prepare_dataset
from surrogate_model.common.metrics import evaluation_metrics, predict_active

X, y = prepare_dataset(train_features, train_labels, features, threshold=threshold)
X_val, y_val = prepare_dataset(
    [val_features], [val_labels], features, threshold=threshold
)

X_train, X_es, y_train, y_es = train_test_split(
    X, y, train_size=0.85, stratify=y, random_state=1000
)

lgbm_tuned = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=25,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=1000,
    deterministic=True,
    force_row_wise=True,
    n_jobs=-1,
)

lgbm_tuned.fit(
    X_train,
    y_train,
    eval_set=[(X_es, y_es)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print(f"Best iteration: {lgbm_tuned.best_iteration_}")
evaluation_metrics(y_val, predict_active(lgbm_tuned, X_val))

/workspaces/lynceus/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 12724, number of negative: 53249
[LightGBM] [Info] Total Bins 211615
[LightGBM] [Info] Number of data points in the train set: 65973, number of used features: 4368
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.192867 -> initscore=-1.431489
[LightGBM] [Info] Start training from score -1.431489
Best iteration: 434


Top %,BEDROC (80.5),BEDROC (160.9),EF,Recall,Num active caught,Num active
f64,f64,f64,f64,f64,i64,i64
1.0,0.451679,0.415673,32.865432,0.328437,313,953
5.0,0.451679,0.415673,13.977356,0.698846,666,953
10.0,0.451679,0.415673,8.510237,0.850997,811,953
25.0,0.451679,0.415673,3.958152,0.989507,943,953


In [7]:
from typing import Any

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.common.data import AFFINITY_COLUMN_NAME, load_and_clean_data
from surrogate_model.common.metrics import evaluation_metrics, predict_active


class AffinityRegressorAdapter:
    """Wraps a regressor predicting docking affinity (lower = better) as a ProbabilisticClassifier."""

    def __init__(self, regressor: lgb.LGBMRegressor, max_affinity: float = 0.0) -> None:
        self.regressor = regressor
        self.max_affinity = max_affinity

    def fit(
        self, X: Any, y_affinity: np.ndarray, **fit_params: Any
    ) -> "AffinityRegressorAdapter":
        y_clipped = np.minimum(
            np.asarray(y_affinity, dtype=np.float32), self.max_affinity
        )
        self.regressor.fit(X, y_clipped, **fit_params)
        return self

    def predict_proba(self, X: Any, *args: Any, **kwargs: Any) -> np.ndarray:
        pred_affinity = np.asarray(self.regressor.predict(X), dtype=np.float64)
        score = -pred_affinity
        return np.column_stack([-score, score])

    def predict(self, X: Any, *args: Any, **kwargs: Any) -> np.ndarray:
        return (self.predict_proba(X)[:, 1] > 0.0).astype(int)

In [8]:
from pathlib import Path

import lightgbm as lgb
import polars as pl
from lynceus_core.features import MOLECULAR_FEATURES
from sklearn.model_selection import train_test_split
from surrogate_model.common.data import (
    flatten_features,
    prepare_dataset,
)
from surrogate_model.common.metrics import evaluation_metrics, predict_active
from surrogate_model.common.threshold import get_validation_threshold

data_dir = Path("data")
rounds = [f"round_{i}" for i in range(1, 5)]

train_features = [str(data_dir / r / "features.parquet") for r in rounds]
train_labels = [str(data_dir / r / "docking.results.parquet") for r in rounds]

val_features = str(data_dir / "validation" / "features.parquet")
val_labels = str(data_dir / "validation" / "docking.results.parquet")

features = list(MOLECULAR_FEATURES)
threshold = get_validation_threshold(val_labels)

# 1. Load continuous affinities for training across rounds 1-4
train_df = pl.concat(
    [
        load_and_clean_data(pl.read_parquet(f_path), pl.read_parquet(l_path))
        for f_path, l_path in zip(train_features, train_labels, strict=True)
    ]
)
X = flatten_features(train_df, features)
y_affinity = train_df[AFFINITY_COLUMN_NAME].to_numpy()

# 2. Load binary validation set (unchanged for BEDROC / EF evaluation)
X_val, y_val = prepare_dataset(
    [val_features], [val_labels], features, threshold=threshold
)

# 3. Fit tuned LGBMRegressor with early stopping
X_tr, X_es, y_tr, y_es = train_test_split(
    X, y_affinity, train_size=0.85, random_state=1000
)

reg = lgb.LGBMRegressor(
    objective="regression",  # Also try objective="huber" if extreme outliers remain
    n_estimators=500,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.25,
    colsample_bynode=0.5,
    reg_alpha=0.5,
    reg_lambda=5.0,
    random_state=1000,
    deterministic=True,
    force_row_wise=True,
    n_jobs=-1,
)

adapter = AffinityRegressorAdapter(reg, max_affinity=0.0)
adapter.fit(
    X_tr,
    y_tr,
    eval_set=[(X_es, np.minimum(y_es, 0.0))],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print(f"Best iteration: {adapter.regressor.best_iteration_}")
evaluation_metrics(y_val, predict_active(adapter, X_val))

/workspaces/lynceus/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Total Bins 211422
[LightGBM] [Info] Number of data points in the train set: 65973, number of used features: 4323
[LightGBM] [Info] Start training from score -7.108331
Best iteration: 500


Top %,BEDROC (80.5),BEDROC (160.9),EF,Recall,Num active caught,Num active
f64,f64,f64,f64,f64,i64,i64
1.0,0.436078,0.393107,31.080409,0.310598,296,953
5.0,0.436078,0.393107,14.2292,0.711438,678,953
10.0,0.436078,0.393107,8.772575,0.87723,836,953
25.0,0.436078,0.393107,3.958152,0.989507,943,953
